In [30]:
##%pip install python-dotenv
##%pip install crewai
##%pip install crewai-tools
##%pip install langchain-openai
##%pip install langchain-tavily
#%pip install tavily-python

# library imports

import os
import json
import logging
from datetime import datetime
from dotenv import load_dotenv

from crewai import Agent, Task, Crew, Process
from crewai_tools import PDFSearchTool
from langchain_openai import ChatOpenAI
from crewai_tools import PDFSearchTool, TavilySearchTool

In [31]:
# Environment variable loading
load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

if not OPENAI_API_KEY:
    raise ValueError("OPENAI_API_KEY is missing. Add it to your .env file.")

if not TAVILY_API_KEY:
    raise ValueError("TAVILY_API_KEY is missing. Add it to your .env file.")

os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
os.environ["TAVILY_API_KEY"] = TAVILY_API_KEY

In [32]:
# PDF Path and logging configuration

PDF_PATH = "trasformer_research_paper-dataset.pdf"

logging.basicConfig(
    filename="agentic_rag_trace.log",
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    force=True
)

def log_event(agent, action, details):
    log_record = {
        "timestamp": datetime.now().isoformat(),
        "agent": agent,
        "action": action,
        "details": details
    }
    logging.info(json.dumps(log_record))

In [33]:
# Create LLMs that crew agents will use for reasoning and decision-making
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

In [34]:
# Create PDFsearch tool and web search tool that agents will use for retrieval

pdf_search_tool = PDFSearchTool(
    pdf=PDF_PATH,
    config=dict(
        llm=dict(
            provider="openai",
            config=dict(
                model="gpt-4o-mini",
                temperature=0
            )
        ),
        embedder=dict(
            provider="openai",
            config=dict(
                model="text-embedding-3-small"
            )
        )
    )
)

web_search_tool = TavilySearchTool()


In [35]:
# Creates a router agent and its job is to decide the route - PDF, web, or LLM - for answering the user's question based on the question's nature.
router_agent = Agent(
    role="Router Agent",
    goal=(
        "Analyze the user's question and decide whether the answer should come "
        "from the PDF, the web, or direct LLM knowledge."
    ),
    backstory=(
        "You are an expert query classifier. You route questions to the correct "
        "retrieval path: PDF for Transformer-paper-specific questions, WEB for "
        "current or external questions, and LLM for general conceptual questions."
    ),
    llm=llm,
    verbose=True,
    allow_delegation=False
)

In [36]:
# Creates retriever agents for PDF and web search
retriever_agent = Agent(
    role="Retriever Agent",

    goal="""
    Retrieve relevant information from PDF or web search and provide
    context to the answer generation agent.
    """,

    backstory="""
    You are a retrieval specialist.
    You know how to use PDF search and web search tools to gather
    relevant information for answering user questions.
    """,

    llm=llm,

    tools=[
        pdf_search_tool,
        web_search_tool
    ],

    verbose=True,

    allow_delegation=False
)

In [37]:
# Create answer generator agent that will take the retrieved information and generate a final answer to the user's question.
answer_generator_agent = Agent(
    role="Answer Generator Agent",
    goal=(
        "Generate a clear, accurate, source-grounded final answer using the "
        "retrieved context."
    ),
    backstory=(
        "You are a careful answer generation agent. You convert retrieved context "
        "into a concise final response, mention the source used, and avoid making "
        "unsupported claims."
    ),
    llm=llm,
    verbose=True,
    allow_delegation=False
)


In [38]:
# Router task function
def create_router_task(question):
    return Task(
        description=f"""
        Analyze the user question and choose exactly one route.

        Available routes:

        PDF:
        Use this route if the question is about the uploaded Transformer paper,
        including attention, self-attention, scaled dot-product attention,
        positional encoding, encoder-decoder architecture, BLEU scores,
        model training, or paper-specific results.

        WEB:
        Use this route if the question asks for current, recent, external,
        public, or internet-based information.

        LLM:
        Use this route if the question is general and does not require
        retrieval from the PDF or the web.

        User question:
        {question}

        Return only one word:
        PDF
        WEB
        LLM
        """,
        expected_output="Exactly one word: PDF, WEB, or LLM.",
        agent=router_agent
    )

In [39]:
# Retriever task function that takes the user's question and the route decided by the router agent, and creates a task for the retriever agent to retrieve relevant context based on the route.
def create_retriever_task(question, route):
    return Task(
        description=f"""
        The Router Agent selected this route:

        {route}

        User question:
        {question}

        Your job is to retrieve relevant context.

        Instructions:

        If the route is PDF:
        - Use the PDF search tool.
        - Retrieve relevant content from the uploaded Transformer paper.
        - Do not use web search.

        If the route is WEB:
        - Use the Tavily web search tool.
        - Retrieve current and relevant web information.
        - Do not use the PDF search tool.

        If the route is LLM:
        - Do not use PDF or web search.
        - Return this exact message:
          No external retrieval required. The question can be answered using general LLM knowledge.

        Return only the retrieved context and source type.
        """,
        expected_output=(
            "Retrieved context with source type: PDF, WEB, or LLM."
        ),
        agent=retriever_agent
    )

In [40]:
# Answer generation task function that takes the user's question and the retrieved context, and creates a task for the answer generator agent to generate a final answer to the user's question.
def create_answer_task(question, route, retrieved_context):
    return Task(
        description=f"""
        Generate the final answer.

        User question:
        {question}

        Route selected:
        {route}

        Retrieved context:
        {retrieved_context}

        Requirements:
        - Provide a clear final answer.
        - Mention the source used: PDF, WEB, or LLM.
        - Include a brief reasoning trace.
        - If the retrieved context is insufficient, say so honestly.
        - Do not invent citations or facts.
        """,
        expected_output=(
            "Final answer with source used and brief reasoning trace."
        ),
        agent=answer_generator_agent
    )

In [41]:
# Three agents orchestration process that takes the user's question, routes it, retrieves context, and generates a final answer.
def run_three_agent_rag(question):
    log_event("System", "Question Received", question)

    # -----------------------------
    # Step 1: Router Agent
    # -----------------------------
    router_task = create_router_task(question)

    router_crew = Crew(
        agents=[router_agent],
        tasks=[router_task],
        process=Process.sequential,
        verbose=True
    )

    route_output = router_crew.kickoff()
    route = str(route_output).strip().upper()

    if "PDF" in route:
        route = "PDF"
    elif "WEB" in route:
        route = "WEB"
    else:
        route = "LLM"

    log_event("Router Agent", "Route Selected", route)

    # -----------------------------
    # Step 2: Retriever Agent
    # -----------------------------
    retriever_task = create_retriever_task(question, route)

    retriever_crew = Crew(
        agents=[retriever_agent],
        tasks=[retriever_task],
        process=Process.sequential,
        verbose=True
    )

    retrieved_context = retriever_crew.kickoff()

    log_event(
        "Retriever Agent",
        "Context Retrieved",
        {
            "route": route,
            "context": str(retrieved_context)
        }
    )

    # -----------------------------
    # Step 3: Answer Generator Agent
    # -----------------------------
    answer_task = create_answer_task(
        question=question,
        route=route,
        retrieved_context=str(retrieved_context)
    )

    answer_crew = Crew(
        agents=[answer_generator_agent],
        tasks=[answer_task],
        process=Process.sequential,
        verbose=True
    )

    final_answer = answer_crew.kickoff()

    log_event(
        "Answer Generator Agent",
        "Final Answer Generated",
        {
            "question": question,
            "route": route,
            "answer": str(final_answer)
        }
    )

    return {
        "question": question,
        "route": route,
        "retrieved_context": str(retrieved_context),
        "final_answer": str(final_answer)
    }

In [42]:
# Test PDF route
result_pdf = run_three_agent_rag(
    "What is scaled dot-product attention in the Transformer paper?"
)

print("Route:", result_pdf["route"])
print("\nFinal Answer:\n")
print(result_pdf["final_answer"])

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name:                                                                                                          │
│  crew                                                                                                           │
│  ID:                                                                                                            │
│  040fd0f3-5f29-4f67-bd2e-c081767fc102                                                                           │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name:                                                                                                          │
│          Analyze the user question and choose exactly one route.                                                │
│                                                                                                                 │
│          Available routes:                                                                                      │
│                                                                                                                 │
│          PDF:                                                                                                   │
│          Use this route if the question is about the uploaded Transformer paper,                                │
│          including attention, self-attention, scaled dot-product attention,                                     │
│          positional encoding, encoder-decoder architecture, BLEU scores,                                        │
│          model training, or paper-specific results.                                                             │
│                                                                                                                 │
│          WEB:                                                                                                   │
│          Use this route if the question asks for current, recent, external,                                     │
│          public, or internet-based information.                                                                 │
│                                                                                                                 │
│          LLM:                                                                                                   │
│          Use this route if the question is general and does not require                                         │
│          retrieval from the PDF or the web.                                                                     │
│                                                                                                                 │
│          User question:                                                                                         │
│          What is scaled dot-product attention in the Transformer paper?                                         │
│                                                                                                                 │
│          Return only one word:                                                                                  │
│          PDF                                                                                                    │
│          WEB                                                                                                    │
│          LLM                                                                                                    │
│                                                                                                                 │
│  ID: a99e05ca-71c7-4794-b8c3-c45617c24459                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Router Agent                                                                                            │
│                                                                                                                 │
│  Task:                                                                                                          │
│          Analyze the user question and choose exactly one route.                                                │
│                                                                                                                 │
│          Available routes:                                                                                      │
│                                                                                                                 │
│          PDF:                                                                                                   │
│          Use this route if the question is about the uploaded Transformer paper,                                │
│          including attention, self-attention, scaled dot-product attention,                                     │
│          positional encoding, encoder-decoder architecture, BLEU scores,                                        │
│          model training, or paper-specific results.                                                             │
│                                                                                                                 │
│          WEB:                                                                                                   │
│          Use this route if the question asks for current, recent, external,                                     │
│          public, or internet-based information.                                                                 │
│                                                                                                                 │
│          LLM:                                                                                                   │
│          Use this route if the question is general and does not require                                         │
│          retrieval from the PDF or the web.                                                                     │
│                                                                                                                 │
│          User question:                                                                                         │
│          What is scaled dot-product attention in the Transformer paper?                                         │
│                                                                                                                 │
│          Return only one word:                                                                                  │
│          PDF                                                                                                    │
│          WEB                                                                                                    │
│          LLM                                                                                                    │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Router Agent                                                                                            │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  PDF                                                                                                            │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name:                                                                                                          │
│                                                                                                                 │
│          Analyze the user question and choose exactly one route.                                                │
│                                                                                                                 │
│          Available routes:                                                                                      │
│                                                                                                                 │
│          PDF:                                                                                                   │
│          Use this route if the question is about the uploaded Transformer paper,                                │
│          including attention, self-attention, scaled dot-product attention,                                     │
│          positional encoding, encoder-decoder architecture, BLEU scores,                                        │
│          model training, or paper-specific results.                                                             │
│                                                                                                                 │
│          WEB:                                                                                                   │
│          Use this route if the question asks for current, recent, external,                                     │
│          public, or internet-based information.                                                                 │
│                                                                                                                 │
│          LLM:                                                                                                   │
│          Use this route if the question is general and does not require                                         │
│          retrieval from the PDF or the web.                                                                     │
│                                                                                                                 │
│          User question:                                                                                         │
│          What is scaled dot-product attention in the Transformer paper?                                         │
│                                                                                                                 │
│          Return only one word:                                                                                  │
│          PDF                                                                                                    │
│          WEB                                                                                                    │
│          LLM                                                                                                    │
│                                                                                                                 │
│  Agent:                                                                                                         │
│  Router Agent                                                                                                   │
│                                                                                                                 │
│                                                        

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Answer Generator Agent                                                                                  │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  Scaled Dot-Product Attention is a specific attention mechanism used in the Transformer model. It operates by   │
│  taking queries and keys of dimension \(d_k\) and values of dimension \(d_v\). The process involves computing   │
│  the dot products of the query with all keys, scaling these dot products by dividing by \(\sqrt{d_k}\), and     │
│  then applying a softmax function to obtain weights for the values. The attention function is computed for a    │
│  set of queries simultaneously, resulting in the output defined as:                                             │
│                                                                                                                 │
│  \[ \text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right) V \]                         │
│                                                                                                                 │
│  This method is more efficient than additive attention, particularly for larger values of \(d_k\), as it can    │
│  leverage optimized matrix multiplication. The scaling helps prevent the softmax function from entering         │
│  regions with very small gradients, which can occur with large dot products.                                    │
│                                                                                                                 │
│  (Source: PDF)                                                                                                  │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name:                                                                                                          │
│                                                                                                                 │
│          Generate the final answer.                                                                             │
│                                                                                                                 │
│          User question:                                                                                         │
│          What is scaled dot-product attention in the Transformer paper?                                         │
│                                                                                                                 │
│          Route selected:                                                                                        │
│          PDF                                                                                                    │
│                                                                                                                 │
│          Retrieved context:                                                                                     │
│          Retrieved context with source type: PDF                                                                │
│                                                                                                                 │
│  Scaled Dot-Product Attention                                                                                   │
│                                                                                                                 │
│  We call our particular attention "Scaled Dot-Product Attention". The input consists of queries and keys of     │
│  dimension dk, and values of dimension dv. We compute the dot products of the query with all keys, divide each  │
│  by √dk, and apply a softmax function to obtain the weights on the values.                                      │
│                                                                                                                 │
│  In practice, we compute the attention function on a set of queries simultaneously, packed together into a      │
│  matrix Q. The keys and values are also packed together into matrices K and V. We compute the matrix of         │
│  outputs as:                                                                                                    │
│                                                                                                                 │
│  Attention(Q, K, V) = softmax(QKT / √dk) V                                                                      │
│                                                                                                                 │
│  The two most commonly used attention functions are additive attention and dot-product (multi­plicative)        │
│  attention. Dot-product attention is identical to our algorithm, except for the scaling factor of 1/√dk.        │
│  Additive attention computes the compatibility function using a feed-forward network with a single hidden       │
│  layer. While the two are similar in theoretical complexity, dot-product attention is much faster and more      │
│  space-efficient in practice, since it can be implemented using highly optimized matrix multiplication code.    │
│                                                                                                                 │
│  While for small values of dk the two mechanisms perfor

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name:                                                                                                          │
│  crew                                                                                                           │
│  ID:                                                                                                            │
│  f22e6771-8644-4493-85ba-44825095469b                                                                           │
│  Final Output: Scaled Dot-Product Attention is a specific attention mechanism used in the Transformer model.    │
│  It operates by taking queries and keys of dimension \(d_k\) and values of dimension \(d_v\). The process       │
│  involves computing the dot products of the query with all keys, scaling these dot products by dividing by      │
│  \(\sqrt{d_k}\), and then applying a softmax function to obtain weights for the values. The attention function  │
│  is computed for a set of queries simultaneously, resulting in the output defined as:                           │
│                                                                                                                 │
│  \[ \text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right) V \]                         │
│                                                                                                                 │
│  This method is more efficient than additive attention, particularly for larger values of \(d_k\), as it can    │
│  leverage optimized matrix multiplication. The scaling helps prevent the softmax function from entering         │
│  regions with very small gradients, which can occur with large dot products.                                    │
│                                                                                                                 │
│  (Source: PDF)                                                                                                  │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Route: PDF

Final Answer:

Scaled Dot-Product Attention is a specific attention mechanism used in the Transformer model. It operates by taking queries and keys of dimension \(d_k\) and values of dimension \(d_v\). The process involves computing the dot products of the query with all keys, scaling these dot products by dividing by \(\sqrt{d_k}\), and then applying a softmax function to obtain weights for the values. The attention function is computed for a set of queries simultaneously, resulting in the output defined as:

\[ \text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right) V \]

This method is more efficient than additive attention, particularly for larger values of \(d_k\), as it can leverage optimized matrix multiplication. The scaling helps prevent the softmax function from entering regions with very small gradients, which can occur with large dot products.

(Source: PDF)


╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [43]:
result_web = run_three_agent_rag(
    "What are the latest trends in Retrieval-Augmented Generation?"
)

print("Route:", result_web["route"])
print("\nFinal Answer:\n")
print(result_web["final_answer"])

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name:                                                                                                          │
│  crew                                                                                                           │
│  ID:                                                                                                            │
│  56c65b44-7995-4279-848a-ea129e1bab28                                                                           │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name:                                                                                                          │
│          Analyze the user question and choose exactly one route.                                                │
│                                                                                                                 │
│          Available routes:                                                                                      │
│                                                                                                                 │
│          PDF:                                                                                                   │
│          Use this route if the question is about the uploaded Transformer paper,                                │
│          including attention, self-attention, scaled dot-product attention,                                     │
│          positional encoding, encoder-decoder architecture, BLEU scores,                                        │
│          model training, or paper-specific results.                                                             │
│                                                                                                                 │
│          WEB:                                                                                                   │
│          Use this route if the question asks for current, recent, external,                                     │
│          public, or internet-based information.                                                                 │
│                                                                                                                 │
│          LLM:                                                                                                   │
│          Use this route if the question is general and does not require                                         │
│          retrieval from the PDF or the web.                                                                     │
│                                                                                                                 │
│          User question:                                                                                         │
│          What are the latest trends in Retrieval-Augmented Generation?                                          │
│                                                                                                                 │
│          Return only one word:                                                                                  │
│          PDF                                                                                                    │
│          WEB                                                                                                    │
│          LLM                                                                                                    │
│                                                                                                                 │
│  ID: 72be7fb5-8671-468d-8f46-16350ee500ac                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Router Agent                                                                                            │
│                                                                                                                 │
│  Task:                                                                                                          │
│          Analyze the user question and choose exactly one route.                                                │
│                                                                                                                 │
│          Available routes:                                                                                      │
│                                                                                                                 │
│          PDF:                                                                                                   │
│          Use this route if the question is about the uploaded Transformer paper,                                │
│          including attention, self-attention, scaled dot-product attention,                                     │
│          positional encoding, encoder-decoder architecture, BLEU scores,                                        │
│          model training, or paper-specific results.                                                             │
│                                                                                                                 │
│          WEB:                                                                                                   │
│          Use this route if the question asks for current, recent, external,                                     │
│          public, or internet-based information.                                                                 │
│                                                                                                                 │
│          LLM:                                                                                                   │
│          Use this route if the question is general and does not require                                         │
│          retrieval from the PDF or the web.                                                                     │
│                                                                                                                 │
│          User question:                                                                                         │
│          What are the latest trends in Retrieval-Augmented Generation?                                          │
│                                                                                                                 │
│          Return only one word:                                                                                  │
│          PDF                                                                                                    │
│          WEB                                                                                                    │
│          LLM                                                                                                    │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Router Agent                                                                                            │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  WEB                                                                                                            │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name:                                                                                                          │
│                                                                                                                 │
│          Analyze the user question and choose exactly one route.                                                │
│                                                                                                                 │
│          Available routes:                                                                                      │
│                                                                                                                 │
│          PDF:                                                                                                   │
│          Use this route if the question is about the uploaded Transformer paper,                                │
│          including attention, self-attention, scaled dot-product attention,                                     │
│          positional encoding, encoder-decoder architecture, BLEU scores,                                        │
│          model training, or paper-specific results.                                                             │
│                                                                                                                 │
│          WEB:                                                                                                   │
│          Use this route if the question asks for current, recent, external,                                     │
│          public, or internet-based information.                                                                 │
│                                                                                                                 │
│          LLM:                                                                                                   │
│          Use this route if the question is general and does not require                                         │
│          retrieval from the PDF or the web.                                                                     │
│                                                                                                                 │
│          User question:                                                                                         │
│          What are the latest trends in Retrieval-Augmented Generation?                                          │
│                                                                                                                 │
│          Return only one word:                                                                                  │
│          PDF                                                                                                    │
│          WEB                                                                                                    │
│          LLM                                                                                                    │
│                                                                                                                 │
│  Agent:                                                                                                         │
│  Router Agent                                                                                                   │
│                                                                                                                 │
│                                                        

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Answer Generator Agent                                                                                  │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  The latest trends in Retrieval-Augmented Generation (RAG) indicate a rapid growth in the market, driven by     │
│  advancements in natural language processing (NLP) and an increasing demand for intelligent AI solutions. RAG   │
│  techniques are becoming pivotal in enhancing large language models, particularly in their application within   │
│  conversational agents. This approach has shown improvements in the robustness of RAG systems, especially in    │
│  handling noisy environments.                                                                                   │
│                                                                                                                 │
│  (Source: WEB)                                                                                                  │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name:                                                                                                          │
│                                                                                                                 │
│          Generate the final answer.                                                                             │
│                                                                                                                 │
│          User question:                                                                                         │
│          What are the latest trends in Retrieval-Augmented Generation?                                          │
│                                                                                                                 │
│          Route selected:                                                                                        │
│          WEB                                                                                                    │
│                                                                                                                 │
│          Retrieved context:                                                                                     │
│          Retrieved context with source type: WEB                                                                │
│                                                                                                                 │
│  The retrieval augmented generation market is growing rapidly due to advancements in natural language           │
│  processing (NLP) and the increasing need for intelligent AI. This report provides an in-depth analysis of the  │
│  latest and advanced techniques in Retrieval Augmented Generation (RAG), a pivotal approach in enhancing large  │
│  language models. RAG has been popularized recently with its application in conversational agents, improving    │
│  the robustness of RAGs in facing noisy environments.                                                           │
│                                                                                                                 │
│          Requirements:                                                                                          │
│          - Provide a clear final answer.                                                                        │
│          - Mention the source used: PDF, WEB, or LLM.                                                           │
│          - Include a brief reasoning trace.                                                                     │
│          - If the retrieved context is insufficient, say so honestly.                                           │
│          - Do not invent citations or facts.                                                                    │
│                                                                                                                 │
│  Agent:                                                                                                         │
│  Answer Generator Agent                                                                                         │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name:                                                                                                          │
│  crew                                                                                                           │
│  ID:                                                                                                            │
│  4e797d97-9f97-4b62-b65d-3986a219db9d                                                                           │
│  Final Output: The latest trends in Retrieval-Augmented Generation (RAG) indicate a rapid growth in the         │
│  market, driven by advancements in natural language processing (NLP) and an increasing demand for intelligent   │
│  AI solutions. RAG techniques are becoming pivotal in enhancing large language models, particularly in their    │
│  application within conversational agents. This approach has shown improvements in the robustness of RAG        │
│  systems, especially in handling noisy environments.                                                            │
│                                                                                                                 │
│  (Source: WEB)                                                                                                  │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Route: WEB

Final Answer:

The latest trends in Retrieval-Augmented Generation (RAG) indicate a rapid growth in the market, driven by advancements in natural language processing (NLP) and an increasing demand for intelligent AI solutions. RAG techniques are becoming pivotal in enhancing large language models, particularly in their application within conversational agents. This approach has shown improvements in the robustness of RAG systems, especially in handling noisy environments.

(Source: WEB)


╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [44]:
# Test LLM Route
result_llm = run_three_agent_rag(
    "Explain RAG in simple terms."
)

print("Route:", result_llm["route"])
print("\nFinal Answer:\n")
print(result_llm["final_answer"])

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name:                                                                                                          │
│  crew                                                                                                           │
│  ID:                                                                                                            │
│  4e34e1b2-b2b1-4c92-9246-a4a2b449fb71                                                                           │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name:                                                                                                          │
│          Analyze the user question and choose exactly one route.                                                │
│                                                                                                                 │
│          Available routes:                                                                                      │
│                                                                                                                 │
│          PDF:                                                                                                   │
│          Use this route if the question is about the uploaded Transformer paper,                                │
│          including attention, self-attention, scaled dot-product attention,                                     │
│          positional encoding, encoder-decoder architecture, BLEU scores,                                        │
│          model training, or paper-specific results.                                                             │
│                                                                                                                 │
│          WEB:                                                                                                   │
│          Use this route if the question asks for current, recent, external,                                     │
│          public, or internet-based information.                                                                 │
│                                                                                                                 │
│          LLM:                                                                                                   │
│          Use this route if the question is general and does not require                                         │
│          retrieval from the PDF or the web.                                                                     │
│                                                                                                                 │
│          User question:                                                                                         │
│          Explain RAG in simple terms.                                                                           │
│                                                                                                                 │
│          Return only one word:                                                                                  │
│          PDF                                                                                                    │
│          WEB                                                                                                    │
│          LLM                                                                                                    │
│                                                                                                                 │
│  ID: 28f040e1-825e-4ce3-925c-f3be5e17e7e4                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Router Agent                                                                                            │
│                                                                                                                 │
│  Task:                                                                                                          │
│          Analyze the user question and choose exactly one route.                                                │
│                                                                                                                 │
│          Available routes:                                                                                      │
│                                                                                                                 │
│          PDF:                                                                                                   │
│          Use this route if the question is about the uploaded Transformer paper,                                │
│          including attention, self-attention, scaled dot-product attention,                                     │
│          positional encoding, encoder-decoder architecture, BLEU scores,                                        │
│          model training, or paper-specific results.                                                             │
│                                                                                                                 │
│          WEB:                                                                                                   │
│          Use this route if the question asks for current, recent, external,                                     │
│          public, or internet-based information.                                                                 │
│                                                                                                                 │
│          LLM:                                                                                                   │
│          Use this route if the question is general and does not require                                         │
│          retrieval from the PDF or the web.                                                                     │
│                                                                                                                 │
│          User question:                                                                                         │
│          Explain RAG in simple terms.                                                                           │
│                                                                                                                 │
│          Return only one word:                                                                                  │
│          PDF                                                                                                    │
│          WEB                                                                                                    │
│          LLM                                                                                                    │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Router Agent                                                                                            │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  LLM                                                                                                            │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name:                                                                                                          │
│                                                                                                                 │
│          Analyze the user question and choose exactly one route.                                                │
│                                                                                                                 │
│          Available routes:                                                                                      │
│                                                                                                                 │
│          PDF:                                                                                                   │
│          Use this route if the question is about the uploaded Transformer paper,                                │
│          including attention, self-attention, scaled dot-product attention,                                     │
│          positional encoding, encoder-decoder architecture, BLEU scores,                                        │
│          model training, or paper-specific results.                                                             │
│                                                                                                                 │
│          WEB:                                                                                                   │
│          Use this route if the question asks for current, recent, external,                                     │
│          public, or internet-based information.                                                                 │
│                                                                                                                 │
│          LLM:                                                                                                   │
│          Use this route if the question is general and does not require                                         │
│          retrieval from the PDF or the web.                                                                     │
│                                                                                                                 │
│          User question:                                                                                         │
│          Explain RAG in simple terms.                                                                           │
│                                                                                                                 │
│          Return only one word:                                                                                  │
│          PDF                                                                                                    │
│          WEB                                                                                                    │
│          LLM                                                                                                    │
│                                                                                                                 │
│  Agent:                                                                                                         │
│  Router Agent                                                                                                   │
│                                                                                                                 │
│                                                        

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Answer Generator Agent                                                                                  │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  Retrieval-Augmented Generation (RAG) is a method that combines two key processes: retrieving information from  │
│  a large database and generating text based on that information. In simple terms, RAG works like this: when     │
│  you ask a question, it first searches for relevant information from a collection of documents or data. Then,   │
│  it uses that information to create a coherent and contextually appropriate response. This approach helps       │
│  improve the accuracy and relevance of the answers provided by AI systems, making them more useful in           │
│  real-world applications.                                                                                       │
│                                                                                                                 │
│  (Source: LLM)                                                                                                  │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name:                                                                                                          │
│                                                                                                                 │
│          Generate the final answer.                                                                             │
│                                                                                                                 │
│          User question:                                                                                         │
│          Explain RAG in simple terms.                                                                           │
│                                                                                                                 │
│          Route selected:                                                                                        │
│          LLM                                                                                                    │
│                                                                                                                 │
│          Retrieved context:                                                                                     │
│          No external retrieval required. The question can be answered using general LLM knowledge.              │
│                                                                                                                 │
│          Requirements:                                                                                          │
│          - Provide a clear final answer.                                                                        │
│          - Mention the source used: PDF, WEB, or LLM.                                                           │
│          - Include a brief reasoning trace.                                                                     │
│          - If the retrieved context is insufficient, say so honestly.                                           │
│          - Do not invent citations or facts.                                                                    │
│                                                                                                                 │
│  Agent:                                                                                                         │
│  Answer Generator Agent                                                                                         │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name:                                                                                                          │
│  crew                                                                                                           │
│  ID:                                                                                                            │
│  043dee78-455f-4745-8f06-9302f7278707                                                                           │
│  Final Output: Retrieval-Augmented Generation (RAG) is a method that combines two key processes: retrieving     │
│  information from a large database and generating text based on that information. In simple terms, RAG works    │
│  like this: when you ask a question, it first searches for relevant information from a collection of documents  │
│  or data. Then, it uses that information to create a coherent and contextually appropriate response. This       │
│  approach helps improve the accuracy and relevance of the answers provided by AI systems, making them more      │
│  useful in real-world applications.                                                                             │
│                                                                                                                 │
│  (Source: LLM)                                                                                                  │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Route: LLM

Final Answer:

Retrieval-Augmented Generation (RAG) is a method that combines two key processes: retrieving information from a large database and generating text based on that information. In simple terms, RAG works like this: when you ask a question, it first searches for relevant information from a collection of documents or data. Then, it uses that information to create a coherent and contextually appropriate response. This approach helps improve the accuracy and relevance of the answers provided by AI systems, making them more useful in real-world applications.

(Source: LLM)


╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [45]:
# View trace logs
with open("agentic_rag_trace.log", "r") as file:
    logs = file.readlines()

for log in logs[-15:]:
    print(log)

2026-06-13 16:00:44,461 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"

2026-06-13 16:00:44,463 - INFO - OpenAI API usage: {'prompt_tokens': 1004, 'completion_tokens': 83, 'total_tokens': 1087}

2026-06-13 16:00:44,493 - INFO - {"timestamp": "2026-06-13T16:00:44.493225", "agent": "Answer Generator Agent", "action": "Final Answer Generated", "details": {"question": "What are the latest trends in Retrieval-Augmented Generation?", "route": "WEB", "answer": "The latest trends in Retrieval-Augmented Generation (RAG) indicate a rapid growth in the market, driven by advancements in natural language processing (NLP) and an increasing demand for intelligent AI solutions. RAG techniques are becoming pivotal in enhancing large language models, particularly in their application within conversational agents. This approach has shown improvements in the robustness of RAG systems, especially in handling noisy environments.\n\n(Source: WEB)"}}

2026-06-13 16:00

In [46]:
# Display final results in a structured format
def display_result(result):
    print("=" * 80)
    print("QUESTION:")
    print(result["question"])
    print("=" * 80)
    print("ROUTE SELECTED:")
    print(result["route"])
    print("=" * 80)
    print("RETRIEVED CONTEXT:")
    print(result["retrieved_context"])
    print("=" * 80)
    print("FINAL ANSWER:")
    print(result["final_answer"])
    print("=" * 80)

display_result(result_pdf)

QUESTION:
What is scaled dot-product attention in the Transformer paper?
ROUTE SELECTED:
PDF
RETRIEVED CONTEXT:
Retrieved context with source type: PDF

Scaled Dot-Product Attention

We call our particular attention "Scaled Dot-Product Attention". The input consists of queries and keys of dimension dk, and values of dimension dv. We compute the dot products of the query with all keys, divide each by √dk, and apply a softmax function to obtain the weights on the values.

In practice, we compute the attention function on a set of queries simultaneously, packed together into a matrix Q. The keys and values are also packed together into matrices K and V. We compute the matrix of outputs as:

Attention(Q, K, V) = softmax(QKT / √dk) V

The two most commonly used attention functions are additive attention and dot-product (multi­plicative) attention. Dot-product attention is identical to our algorithm, except for the scaling factor of 1/√dk. Additive attention computes the compatibility functi